<a href="https://colab.research.google.com/github/timraiswell/ai-engineer/blob/main/04-evals/03-regression-evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regression Evals

**Goal:** Turn the golden set and judge into a CI-style harness: run, save, diff, and gate.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client; `sentence-transformers`, `numpy` are used by this notebook.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [1]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git" sentence-transformers numpy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.4 MB/s eta 0:00:00


In [2]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


> **🔵 Hitting a rate limit? Switch models.** This is the heaviest notebook in the section. It runs the full golden set **twice** (baseline + candidate), each a generation call *plus* a judge call per case. Groq's free tier caps tokens-per-minute and tokens-per-day **per model**, so a `429` here is easy to hit. Two knobs, each with its own budget:
> ```python
> client, MODEL = setup(model='openai/gpt-oss-20b')   # the system-under-test (generation)
> JUDGE_MODEL = 'qwen/qwen3.6-27b'                     # set in the judge cell below
> ```
> Keep the judge a **different** model from the system under test. Pick models with headroom from the [Groq rate-limits page](https://console.groq.com/docs/rate-limits); for a verification pass any capable chat models work (avoid the agentic `groq/compound*` ones). Cheaper still: slice the set with `GOLDEN_SET[:4]` while you're just checking the harness runs.

## Evals as CI

You change a prompt, a chunk size, or a model version. Something breaks. The only question is whether the eval tells you, or a customer does.

That's the whole frame for this notebook: the golden set (notebook 01) plus the judge (notebook 02) become a test suite you run before every change ships. The mechanics are identical to the CI you already know: run the suite, save the results, diff against the last green run, block the merge if a gate fails. The only new part is that individual cases are noisy, so the diffing and gating logic has to tolerate flakiness without ignoring real regressions.


## The system under test

The next two cells rebuild the RAG system from section 03 in compact form: download and clean ten IETF RFCs, chunk them by section, embed with a small local model, retrieve by cosine similarity, and answer with citations.

This is a deliberate copy, not an import. Each notebook has to run top-to-bottom in a fresh Colab runtime, and self-containment beats DRY for teaching material. If the details are unfamiliar, work through `03-rag/` first; here they're just the thing we're evaluating.


In [3]:
# Compact copy of the 03-rag pipeline. Self-containment over DRY: this notebook must run
# standalone in Colab. See 03-rag/ for the full walkthrough of every design decision here.
import os
import re
import urllib.request

RFCS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
os.makedirs('data/rfc', exist_ok=True)
for n in RFCS:
    path = f'data/rfc/rfc{n}.txt'
    if not os.path.exists(path):  # skip if cached from an earlier notebook
        urllib.request.urlretrieve(f'https://www.rfc-editor.org/rfc/rfc{n}.txt', path)

def clean_rfc(text):
    """Strip form feeds and the page header/footer lines RFC txt files carry."""
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.search(r'\[Page \d+\]\s*$', l)          # footers
             and not re.match(r'^\s*RFC \d+.*\d{4}\s*$', l)]   # headers
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines))

def chunk_rfc(text, rfc, max_chars=2000):
    """Section-aware chunking: split on numbered headings, then cap chunk size."""
    parts = re.split(r'\n(?=\d+(?:\.\d+)*\.?\s+[A-Z])', text)
    chunks = []
    for part in parts:
        part = part.strip()
        while len(part) > max_chars:
            cut = part.rfind('\n\n', 0, max_chars)
            cut = cut if cut > 200 else max_chars
            chunks.append({'rfc': rfc, 'text': part[:cut].strip()})
            part = part[cut:].strip()
        if len(part) > 100:
            chunks.append({'rfc': rfc, 'text': part})
    return chunks

chunks = []
for n in RFCS:
    with open(f'data/rfc/rfc{n}.txt') as f:
        chunks += chunk_rfc(clean_rfc(f.read()), n)
print(f'{len(chunks)} chunks from {len(RFCS)} RFCs')

1662 chunks from 10 RFCs


In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
emb = embedder.encode([c['text'] for c in chunks],
                      normalize_embeddings=True, show_progress_bar=True)

def retrieve(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = emb @ q  # cosine similarity: vectors are unit-normalized
    return [chunks[i] for i in np.argsort(-scores)[:k]]

def answer_question(question, k=5):
    """Retrieve top-k chunks, answer with citations. Returns (answer, hits)."""
    hits = retrieve(question, k)
    context = '\n\n'.join(f"[RFC {h['rfc']}]\n{h['text']}" for h in hits)
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[
            {'role': 'system',
             'content': ('Answer using ONLY the provided RFC excerpts. Cite the RFC number for '
                         'each claim, e.g. (RFC 9110). If the excerpts do not contain the answer, '
                         'say plainly that the corpus does not cover it. Do not guess.')},
            {'role': 'user', 'content': f'{context}\n\nQuestion: {question}'},
        ],
    )
    return resp.choices[0].message.content, hits

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/52 [00:00<?, ?it/s]

Self-contained copies of the golden set and judge from notebooks 01 and 02 (see those for the design reasoning).


In [5]:
# The golden set: ~15 cases written against the actual corpus. Each case pins down
# what a correct answer must contain (must_mention) and which RFC the retriever
# should surface (source_rfc). Unanswerable cases have source_rfc=None -- the correct
# behavior there is to decline, and hallucinating an answer is a hard failure.
GOLDEN_SET = [
    {'question': 'How does the TCP three-way handshake establish a connection?',
     'reference_answer': 'The initiator sends a SYN carrying its initial sequence number; '
                         'the peer responds with SYN-ACK carrying its own sequence number and '
                         'acknowledging the first; the initiator replies with an ACK. Both '
                         'sides now have synchronized sequence numbers.',
     'source_rfc': 793, 'must_mention': ['SYN', 'ACK', 'sequence'], 'unanswerable': False},
    {'question': 'What does the HTTP 404 status code mean?',
     'reference_answer': '404 Not Found: the origin server did not find a current '
                         'representation for the target resource, or is not willing to '
                         'disclose that one exists.',
     'source_rfc': 9110, 'must_mention': ['404', 'not found'], 'unanswerable': False},
    {'question': 'What are the four roles defined in the OAuth 2.0 framework?',
     'reference_answer': 'Resource owner, resource server, client, and authorization server.',
     'source_rfc': 6749,
     'must_mention': ['resource owner', 'client', 'authorization server', 'resource server'],
     'unanswerable': False},
    {'question': 'What is the difference between an A record and a CNAME record in DNS?',
     'reference_answer': 'An A record maps a name to a 32-bit IPv4 host address; a CNAME '
                         'record maps an alias to the canonical name of its target.',
     'source_rfc': 1035, 'must_mention': ['address', 'canonical'], 'unanswerable': False},
    {'question': 'How does a client upgrade an HTTP connection to a WebSocket connection?',
     'reference_answer': 'The client sends a GET request with Upgrade: websocket and '
                         'Connection: Upgrade headers plus a Sec-WebSocket-Key; the server '
                         'replies 101 Switching Protocols with a Sec-WebSocket-Accept value '
                         'derived from the key.',
     'source_rfc': 6455, 'must_mention': ['Upgrade', '101', 'Sec-WebSocket-Key'],
     'unanswerable': False},
    {'question': 'Which SMTP commands send a mail message, and in what order?',
     'reference_answer': 'MAIL FROM identifies the sender, one or more RCPT TO commands name '
                         'recipients, then DATA introduces the message text, terminated by a '
                         'line containing only a period.',
     'source_rfc': 5321, 'must_mention': ['MAIL', 'RCPT', 'DATA'], 'unanswerable': False},
    {'question': 'What is the purpose of the Time to Live field in the IPv4 header?',
     'reference_answer': 'TTL is an upper bound on datagram lifetime. Every module that '
                         'processes the datagram decrements it, and the datagram is discarded '
                         'when it reaches zero, so packets cannot loop forever.',
     'source_rfc': 791, 'must_mention': ['zero', 'discard'], 'unanswerable': False},
    {'question': 'What is the AS_PATH attribute in BGP and why does it matter?',
     'reference_answer': 'AS_PATH records the sequence of autonomous systems a route '
                         'advertisement has traversed. BGP uses it for loop detection (a '
                         'router rejects routes containing its own AS number) and in route '
                         'selection.',
     'source_rfc': 4271, 'must_mention': ['autonomous system', 'loop'], 'unanswerable': False},
    {'question': 'How does HTTP/2 carry multiple requests over a single connection?',
     'reference_answer': 'HTTP/2 multiplexes exchanges as independent streams over one '
                         'connection. Each frame carries a stream identifier, so concurrent '
                         'requests and responses interleave without blocking each other.',
     'source_rfc': 7540, 'must_mention': ['stream', 'frame'], 'unanswerable': False},
    {'question': 'Which HTTP methods are defined as idempotent?',
     'reference_answer': 'PUT and DELETE, plus all safe methods: GET, HEAD, OPTIONS, TRACE.',
     'source_rfc': 9110, 'must_mention': ['PUT', 'DELETE', 'GET'], 'unanswerable': False},
    {'question': 'Why does a TCP connection enter the TIME-WAIT state, and for how long?',
     'reference_answer': 'The side that closes actively waits in TIME-WAIT for twice the '
                         'maximum segment lifetime (2*MSL), so delayed segments from the old '
                         'incarnation die off before the socket pair can be reused.',
     'source_rfc': 793, 'must_mention': ['TIME-WAIT', 'MSL'], 'unanswerable': False},
    {'question': 'Walk through the OAuth 2.0 authorization code grant flow.',
     'reference_answer': 'The client redirects the resource owner to the authorization '
                         'server; after authentication and consent the server redirects back '
                         'with an authorization code; the client exchanges the code, with its '
                         'own credentials, at the token endpoint for an access token.',
     'source_rfc': 6749, 'must_mention': ['authorization code', 'access token', 'redirect'],
     'unanswerable': False},
    {'question': 'What does HTTP status 503 indicate, and which header can accompany it?',
     'reference_answer': '503 Service Unavailable: the server is currently unable to handle '
                         'the request, due to overload or maintenance. It may send Retry-After '
                         'to indicate how long to wait.',
     'source_rfc': 9110, 'must_mention': ['503', 'Retry-After'], 'unanswerable': False},
    # --- deliberately unanswerable: correct behavior is to decline ---
    {'question': 'How does QUIC combine the transport and cryptographic handshakes?',
     'reference_answer': 'Declines: QUIC (RFC 9000) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'What cipher suites does TLS 1.3 define?',
     'reference_answer': 'Declines: TLS 1.3 (RFC 8446) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'Which port does IMAP use, and how does a client select a mailbox?',
     'reference_answer': 'Declines: IMAP is not in the corpus (SMTP is the only mail '
                         'protocol here).',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
]
print(len(GOLDEN_SET), 'cases,',
      sum(c['unanswerable'] for c in GOLDEN_SET), 'unanswerable')


16 cases, 3 unanswerable


In [6]:
def keyword_coverage(case, answer):
    if not case['must_mention']:
        return None
    a = answer.lower()
    return sum(kw.lower() in a for kw in case['must_mention']) / len(case['must_mention'])

DECLINE_MARKERS = ['does not cover', "doesn't cover", 'not covered', 'do not contain',
                   'does not contain', 'not in the corpus', 'cannot answer',
                   'no information', 'not addressed', 'do not include']

def declined(answer):
    a = answer.lower()
    return any(m in a for m in DECLINE_MARKERS)


In [7]:
import json

# Judge model: the small tier, not the big one. Two reasons. (1) Develop cheap, eval on
# target: while you iterate on the eval harness itself you will rerun it constantly, so use
# the cheap model; before a real release decision, rerun the judging pass with a
# stronger model or hand-review. (2) A different model than the system under test
# avoids self-preference bias (more on that below).
JUDGE_MODEL = 'openai/gpt-oss-20b'

JUDGE_TOOL = {
    'type': 'function',
    'function': {
        'name': 'grade_answer',
        'description': 'Record your grade for the candidate answer.',
        'parameters': {
            'type': 'object',
            'properties': {
                'reasoning': {'type': 'string',
                              'description': 'Compare candidate to reference point by point, '
                                             'BEFORE deciding any grade. 2-4 sentences.'},
                'faithful': {'type': 'boolean',
                             'description': 'True if every claim in the candidate is supported '
                                            'by the reference (nothing fabricated).'},
                'complete': {'type': 'boolean',
                             'description': 'True if the candidate covers every key point of '
                                            'the reference.'},
                'score': {'type': 'integer',
                          'description': 'Overall 1-5, anchored to the rubric.'},
            },
            'required': ['reasoning', 'faithful', 'complete', 'score'],
        },
    },
}

JUDGE_SYSTEM = """You grade answers from a retrieval-augmented QA system against a reference answer.

Rubric -- anchor your score to these definitions and use the full range:
1 = wrong or fabricated: contradicts the reference or invents facts
2 = mostly wrong: a relevant fragment, but the main point is missing or incorrect
3 = partially correct: main point present, but a real gap or a minor unsupported claim
4 = correct: matches the reference on every key point; only minor omissions
5 = correct and complete: nothing missing, nothing unsupported

If the reference says the correct behavior is to decline, a candidate that declines
scores 5 and a candidate that invents an answer scores 1.

Write your reasoning FIRST, then decide the score. Judge substance, not style:
a short correct answer outranks a long vague one. Extra length earns nothing."""

from groq import BadRequestError

def judge(question, reference, candidate, tries=3):
    """Grade one answer, retrying Groq's intermittent tool_use_failed (see notebook 02).
    Returns {reasoning, faithful, complete, score}; a sentinel score 0 if it never
    produced a valid call, so a flaky grade shows as a failed case, not a crashed run."""
    for _ in range(tries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL, max_tokens=700,
                tools=[JUDGE_TOOL],
                tool_choice={'type': 'function', 'function': {'name': 'grade_answer'}},
                messages=[
                    {'role': 'system', 'content': JUDGE_SYSTEM},
                    {'role': 'user', 'content':
                     f'Question: {question}\n\nReference answer:\n{reference}'
                     f'\n\nCandidate answer:\n{candidate}'},
                ],
            )
            return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)
        except BadRequestError as e:
            if 'tool_use_failed' in str(e):
                continue
            raise
    return {'reasoning': '(no valid tool call after retries)',
            'faithful': False, 'complete': False, 'score': 0}

## The harness: `run_eval(config)` → saved, timestamped results

One function takes a system configuration, runs the full set, and returns a results dict: per-case rows plus aggregate metrics (`hit_rate`, `keyword_coverage`, `judge_score_mean`, and a hard flag for the unanswerable set). Every run is saved as timestamped JSON in `eval_runs/`.

Should `eval_runs/` be gitignored or committed? Both are defensible. Committing gives you history and blame for free (a results file next to the commit that produced it) at the cost of noisy diffs and answers-in-repo. Gitignoring keeps the repo clean but means you need somewhere else to keep history: an artifact store, or eventually a hosted eval platform. A reasonable middle ground is to gitignore the per-run files and commit a small `baseline.json` that gates compare against.


In [8]:
import json
import time

def run_eval(top_k=5, label='baseline'):
    """Run the golden set against the RAG system with the given config.
    Cost note: one generation call (120B) + one judge call (20B) per case."""
    rows = []
    for case in GOLDEN_SET:
        answer, hits = answer_question(case['question'], k=top_k)
        hit = (case['source_rfc'] in [h['rfc'] for h in hits]
               if case['source_rfc'] else None)
        cov = keyword_coverage(case, answer)
        grade = judge(case['question'], case['reference_answer'], answer)
        if case['unanswerable']:
            passed = declined(answer) or grade['score'] >= 4  # judge understands declines
        else:
            passed = bool(hit) and cov >= 0.6 and grade['score'] >= 3
        rows.append({'question': case['question'],
                     'unanswerable': case['unanswerable'],
                     'hit': hit,
                     'coverage': None if cov is None else round(cov, 2),
                     'score': grade['score'],
                     'passed': passed,
                     'answer': answer})
    answerable = [r for r in rows if not r['unanswerable']]
    unanswerable = [r for r in rows if r['unanswerable']]
    results = {
        'label': label,
        'timestamp': time.strftime('%Y%m%dT%H%M%S'),
        'config': {'top_k': top_k, 'model': MODEL, 'judge_model': JUDGE_MODEL},
        'aggregate': {
            'hit_rate': sum(r['hit'] for r in answerable) / len(answerable),
            'keyword_coverage': sum(r['coverage'] for r in answerable) / len(answerable),
            'judge_score_mean': sum(r['score'] for r in answerable) / len(answerable),
            'unanswerable_ok': all(r['passed'] for r in unanswerable),
            'pass_rate': sum(r['passed'] for r in rows) / len(rows),
        },
        'cases': rows,
    }
    os.makedirs('eval_runs', exist_ok=True)
    path = f"eval_runs/{results['timestamp']}_{label}.json"
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'saved {path}')
    for k, v in results['aggregate'].items():
        print(f'  {k:18}: {v if isinstance(v, bool) else round(v, 3)}')
    return results

In [9]:
# Baseline: the system as built in section 03, top_k=5.
baseline = run_eval(top_k=5, label='baseline_k5')


saved eval_runs/20260831T225107_baseline_k5.json
  hit_rate          : 1.0
  keyword_coverage  : 0.846
  judge_score_mean  : 4.077
  unanswerable_ok   : True
  pass_rate         : 0.812


## Ship a change, watch it break

Now the demo that makes the whole notebook concrete. We "ship" a plausible-looking change: dropping `top_k` from 5 to 1 to cut context tokens and latency. It's the kind of change that sails through code review. The code is correct, the demo query still works, and it saves money. The eval is the only thing standing between it and production.


In [10]:
# The 'optimization': retrieve only the single best chunk.
# Another 2 calls per case (120B generation + 20B judge).
candidate = run_eval(top_k=1, label='candidate_k1')

saved eval_runs/20260831T225225_candidate_k1.json
  hit_rate          : 0.846
  keyword_coverage  : 0.5
  judge_score_mean  : 2.462
  unanswerable_ok   : True
  pass_rate         : 0.375


In [11]:
def compare_runs(a, b):
    """Case-by-case diff of two eval runs (same golden set, same order)."""
    print(f"{a['label']}  ->  {b['label']}")
    print(f"{'metric':18} {'before':>8} {'after':>8}")
    for key, va in a['aggregate'].items():
        vb = b['aggregate'][key]
        fa = str(va) if isinstance(va, bool) else f'{va:.3f}'
        fb = str(vb) if isinstance(vb, bool) else f'{vb:.3f}'
        flag = '  <-- ' if fa != fb else ''
        print(f'{key:18} {fa:>8} {fb:>8}{flag}')

    regressions, improvements, unchanged = [], [], []
    for ca, cb in zip(a['cases'], b['cases']):
        if ca['passed'] and not cb['passed']:
            regressions.append(cb)
        elif not ca['passed'] and cb['passed']:
            improvements.append(cb)
        else:
            unchanged.append(cb)

    print(f'\nregressions ({len(regressions)}):  # passed before, fails now')
    for r in regressions:
        print(f"  FAIL hit={r['hit']} cov={r['coverage']} score={r['score']} | "
              f"{r['question'][:55]}")
    print(f'improvements ({len(improvements)}):')
    for r in improvements:
        print(f"  PASS | {r['question'][:60]}")
    print(f'unchanged: {len(unchanged)}')
    return regressions, improvements

regressions, improvements = compare_runs(baseline, candidate)


baseline_k5  ->  candidate_k1
metric               before    after
hit_rate              1.000    0.846  <-- 
keyword_coverage      0.846    0.500  <-- 
judge_score_mean      4.077    2.462  <-- 
unanswerable_ok        True     True
pass_rate             0.812    0.375  <-- 

regressions (7):  # passed before, fails now
  FAIL hit=True cov=0.5 score=5 | What does the HTTP 404 status code mean?
  FAIL hit=True cov=0.5 score=1 | What is the difference between an A record and a CNAME 
  FAIL hit=True cov=0.33 score=2 | How does a client upgrade an HTTP connection to a WebSo
  FAIL hit=True cov=0.0 score=1 | Which SMTP commands send a mail message, and in what or
  FAIL hit=True cov=0.0 score=1 | How does HTTP/2 carry multiple requests over a single c
  FAIL hit=False cov=1.0 score=5 | Which HTTP methods are defined as idempotent?
  FAIL hit=True cov=0.67 score=1 | Walk through the OAuth 2.0 authorization code grant flo
improvements (0):
unchanged: 9


## Thresholds and gates

A diff needs a policy to become CI. There are two kinds of gates, deliberately different in strictness, and the split is the whole idea:

| Gate type | Applies to | Trigger logic | Tolerates | Example |
|---|---|---|---|---|
| **Budget gate** | noisy aggregates (mean judge score, hit-rate) | block only when the drop exceeds a budget | single-case flakiness | "mean score may not drop > 0.5" |
| **Zero-tolerance gate** | must-never-ship behaviors | any single occurrence hard-fails | nothing | "any hallucination on the unanswerable set" |

- **Budget gates** for the noisy aggregates: mean judge score may not drop more than 0.5, hit-rate not more than 10 points. Individual cases flake (the judge is a model; generation is nondeterministic), so single-case failures shouldn't block a merge. Sustained drops should.
- **Zero-tolerance gates** for the failure modes that must never ship: any hallucinated answer on the unanswerable set is a hard fail, regardless of how good the averages look. A system that scores 4.8 but confidently invents answers to questions it can't support is worse than one that scores 4.2 and knows its limits.

In [12]:
def check_gates(candidate, baseline, max_score_drop=0.5, max_hit_drop=0.10):
    """Return a list of gate failures (empty = safe to ship)."""
    failures = []
    c, b = candidate['aggregate'], baseline['aggregate']
    if b['judge_score_mean'] - c['judge_score_mean'] > max_score_drop:
        failures.append(f"judge_score_mean dropped "
                        f"{b['judge_score_mean'] - c['judge_score_mean']:.2f} "
                        f"(budget: {max_score_drop})")
    if b['hit_rate'] - c['hit_rate'] > max_hit_drop:
        failures.append(f"hit_rate dropped {b['hit_rate'] - c['hit_rate']:.2f} "
                        f"(budget: {max_hit_drop})")
    if not c['unanswerable_ok']:
        failures.append('hallucinated on the unanswerable set (zero tolerance -- '
                        'hard fail regardless of averages)')
    return failures

failures = check_gates(candidate, baseline)
if failures:
    print('GATE FAILED -- do not ship:')
    for f in failures:
        print('  -', f)
else:
    print('all gates passed')

# In CI you would exit nonzero instead of printing:
#   import sys; sys.exit(1 if failures else 0)


GATE FAILED -- do not ship:
  - judge_score_mean dropped 1.62 (budget: 0.5)
  - hit_rate dropped 0.15 (budget: 0.1)


## Wiring it into CI

The harness is already CI-shaped: `run_eval()` produces a JSON artifact, `check_gates()` returns pass/fail. Move the notebook cells into a script (`scripts/run_eval.py`) that loads a committed `eval_runs/baseline.json`, runs the candidate config, and exits nonzero on gate failure. A minimal GitHub Actions step:

```yaml
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - run: pip install groq sentence-transformers numpy
      - run: python scripts/run_eval.py --baseline eval_runs/baseline.json
        env:
          GROQ_API_KEY: ${{ secrets.GROQ_API_KEY }}
      - uses: actions/upload-artifact@v4
        if: always()
        with: {name: eval-results, path: eval_runs/}
```

Practical notes: run it on PRs that touch prompts, retrieval code, or model config (a `paths:` filter keeps it off unrelated changes); the ~32 API calls cost cents and a few minutes, well inside PR-feedback budget; and when someone legitimately improves the system, updating the committed baseline *is* the review artifact, since the diff shows exactly which cases got better.

## Where the frameworks come in

This file-based harness IS the concept; there's nothing conceptually missing. What hosted eval platforms (Braintrust, LangSmith, Langfuse) and OSS frameworks (promptfoo, and the metric libraries like ragas) add is the operational layer: dashboards instead of printed tables, run history with trends instead of a directory of JSON files, per-case blame across weeks of runs, team review queues for the human-labeling loop, and prebuilt CI integrations. Adopt one when the team grows past you, when more than one person needs to see the runs and the JSON directory stops scaling. You'll evaluate those tools well precisely because you know what they're automating.


## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Run the golden set before every prompt/chunk/model change | Ship the change and let a customer find the regression |
| Read the **case-level diff**: which questions broke and why | Watch only the aggregate delta ("score dropped 0.8") |
| Budget gates for noisy aggregates; zero-tolerance for must-never-ship | One global threshold; block merges on single-case flakiness |
| Set thresholds just above the measured noise floor | Guess thresholds; get flappy or blind gates |
| Keep it cheap/fast enough to run in CI on every relevant PR | A 500-case suite you run monthly, catching regressions late |
| Gate cost/latency too, not just quality | "Fixes" that stuff more context in to nudge the score up |

(The full stack-wide list lives in [docs/best-practices-and-anti-patterns.md](https://github.com/calmrocks/ai-engineer-notebooks/blob/main/docs/best-practices-and-anti-patterns.md).)

## Exercises

1. **Regress the chunker instead of top_k.** Add a `chunk_max_chars` parameter to the pipeline, rebuild chunks and embeddings at 400 chars (a genuinely worse chunker that splits sections mid-thought), and `run_eval` it against the baseline. Does it produce the same regression fingerprint as `top_k=1`, or do different cases break?
2. **Measure eval flakiness.** Run `run_eval(top_k=5)` three times with no changes and diff the runs pairwise. How many cases flip with zero code change? Set your gate budgets (`max_score_drop`, `max_hit_drop`) just above the observed noise floor. That's how the thresholds should actually be chosen.
3. **Build the CI script.** Extract the harness into `scripts/run_eval.py` with argparse (`--top-k`, `--baseline`, `--save-baseline`), exiting 0/1 from `check_gates`. Run it locally twice: once to write the baseline, once to gate a `--top-k 1` candidate against it.
4. **Add a cost/latency column.** Record `usage.input_tokens`, `usage.output_tokens`, and wall-clock time per case in `run_eval`, and add a gate: mean input tokens may not grow more than 25% versus baseline. Quality gates without cost gates invite "fixes" that just stuff more context in.
